# ScamScope — give a phishing agent one new tool

**AWS Builder Center — "Give Your Agent a Tool" challenge**

This notebook walks through ScamScope end to end: the baseline phishing-analysis agent, the one new tool we gave it (`check_url_reputation`, backed by an AWS Lambda that queries URLhaus threat intelligence), and a live **before / after** on the same suspicious message.

**Stack:** Strands Agents SDK · Amazon Bedrock (Amazon Nova Pro) · AWS Lambda · AWS Secrets Manager · AWS IAM · Amazon CloudWatch · `us-east-1`

---
### Before you run

1. Rebuild the virtual environment (Python 3.12) and install deps:
   ```bash
   cd "/path/to/scamscope-ai"
   python3.12 -m venv .venv
   ./.venv/bin/pip install -r requirements-dev.txt ipykernel
   ```
2. Register the venv as a Jupyter kernel and select it for this notebook:
   ```bash
   ./.venv/bin/python -m ipykernel install --user --name scamscope
   ```
3. Have AWS credentials configured for an identity that can call Bedrock and invoke the `check_url_reputation` Lambda in `us-east-1` (`aws configure` / `aws sso login`).
4. The Phase-2 tool must be deployed (see `infra/DEPLOY.md`). Phase-1 (baseline) cells work without it.

> The agent cells make **real Amazon Bedrock calls** and the tooled cell makes a **real AWS Lambda invocation**, which may incur small AWS costs.

## 0. Setup — point Python at the source tree and set the region

In [ ]:
import os, sys, pathlib

# Resolve the project root (this notebook lives in demo/notebook/).
ROOT = pathlib.Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src" / "scamscope").is_dir():
    ROOT = ROOT.parent
SRC = ROOT / "src"
assert SRC.is_dir(), f"Could not find src/ under {ROOT}. Run this notebook from inside the scamscope-ai repo."
sys.path.insert(0, str(SRC))

# Pin region (Nova Pro + Lambda both live in us-east-1).
os.environ.setdefault("AWS_REGION", "us-east-1")
os.environ.setdefault("SCAMSCOPE_REGION", "us-east-1")

print("project root:", ROOT)
print("region:", os.environ["SCAMSCOPE_REGION"])

## 1. The suspicious message

A classic phishing message: urgency, an account-closure threat, a request to "verify your identity" via a link. Replace `MAL_URL` with a currently-online malware URL from the [URLhaus feed](https://urlhaus.abuse.ch/) if the sample has gone offline — the `threat_match` only fires on a URL URLhaus still lists as online.

In [ ]:
MAL_URL = "http://221.15.172.85:48113/bin.sh"  # real online malware URL at demo time; swap for a fresh one if offline

MESSAGE = (
    "URGENT: Your account has been temporarily suspended due to unusual "
    "sign-in activity. To avoid permanent closure within 24 hours, verify "
    "your identity immediately here:\n"
    f"{MAL_URL}\n"
    "Failure to confirm will result in loss of access. — Security Team"
)
print(MESSAGE)

## 2. BEFORE — the baseline agent (no tool)

The baseline agent runs on Amazon Nova Pro via the Strands Agents SDK. It reads the message and explains the phishing warning signs — but on the **link** it can only *reason about how it looks*. It has no way to verify whether the URL is actually malicious.

In [ ]:
from scamscope.agent_baseline import analyse as analyse_baseline

before = analyse_baseline(MESSAGE)
print(before)

Notice the hedge on the URL — something like *"does not appear to be associated with a legitimate domain."* That is a guess. (Amazon Bedrock's content filter may also truncate the tail when it sees the raw hostile URL.)

## 3. The one new tool — `check_url_reputation`

The tool is a Strands `@tool` wrapper around an **AWS Lambda** function. The wrapper invokes the Lambda with `boto3`; the Lambda reads a URLhaus Auth-Key from **AWS Secrets Manager** and queries URLhaus — sending only the URL **string**, so it **never fetches the hostile link**.

Let's call the tool directly (outside the agent) to see its structured verdict.

In [ ]:
import json
from scamscope.tools.url_reputation import check_url_reputation

# @tool-decorated callables expose the underlying function; call it directly here.
_fn = getattr(check_url_reputation, "func", None) or getattr(check_url_reputation, "__wrapped__", check_url_reputation)
verdict = _fn(MAL_URL)
print(json.dumps(verdict, indent=2))

A `status` of `threat_match` with `source: urlhaus` is hard evidence the link is malicious. `no_match` means URLhaus has no record — **not** proof the link is safe. `invalid_input` / `lookup_failed` are the safe degraded states.

## 4. AFTER — the tooled agent

Same message, same model — but now the agent has the tool. It reasons, decides to verify the URL, calls `check_url_reputation`, and folds the real URLhaus verdict into a calibrated judgement.

In [ ]:
from scamscope.agent_tooled import analyse as analyse_tooled

after = analyse_tooled(MESSAGE)
print(after)

## 5. Before / after, side by side

In [ ]:
line = "=" * 70
print(line); print("BEFORE  (baseline — no tool, reasoning only)"); print(line)
print(before)
print(); print(line); print("AFTER   (tooled — check_url_reputation via AWS Lambda)"); print(line)
print(after)

## 6. Architecture

```
User (suspicious message)
      |
      v
Amazon Bedrock -- Nova Pro -- Strands Agent
      |  decides to verify a URL
      v
@tool check_url_reputation --boto3 invoke--> AWS Lambda
                                               | Auth-Key  --> AWS Secrets Manager
                                               | lookup (string only) --> URLhaus (external)
                                               v
                                      structured verdict + evidence
      ^                                        |
      +---------------- cited answer <---------+

+ AWS IAM (least-privilege role: read one secret, write own logs)
+ Amazon CloudWatch (invocation proof)
```

A rendered diagram is at `demo/article/architecture.png` (editable source: `demo/article/architecture.drawio`).

---
### Why it matters

One well-chosen tool changed the *kind* of answer the agent can give — from a plausible opinion about a link to a **verified** call — without touching the model. The tool is safe by design (it never opens the hostile URL), the secret never lands in code, and the IAM role can do exactly two things. The same Lambda + Secrets Manager + least-privilege-IAM pattern drops into any agent that reads untrusted text.

*Repo: (add your public GitHub link) · Demo video: (add your YouTube link)*